In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# insert data here
df = pd.read_csv("Traffic_Volume_Counts_20251111.csv")
# print(df.shape)
# print(df.columns)
# print(df.head())

print("\nMissing values:\n", df.isna().sum())

print("\nData types:\n", df.dtypes)

# identify hours
hour_columns = df.columns[7:]

df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors="coerce")
hourly_cols = [col for col in df.columns if ":" in col]
df_long = df.melt(
    id_vars=["ID", "SegmentID", "Roadway Name", "From", "To", "Direction", "Date"],
    value_vars=hourly_cols,
    var_name="Hour",
    value_name="Traffic_Count"
)
def parse_hour(h):
    try:
        time_part = h.split('-')[0].strip()
        return pd.to_datetime(time_part, format="%I:%M %p").hour
    except:
        return np.nan
df_long["Hour_num"] = df_long["Hour"].apply(parse_hour)
df_long.dropna(subset=["Traffic_Count", "Hour_num"], inplace=True)
df_long["Date"] = pd.to_datetime(df_long["Date"], errors="coerce")
df_long["Hour_num"] = pd.to_numeric(df_long["Hour_num"], errors="coerce")

df_long = df_long.dropna(subset=["Date", "Hour_num"])
df_long["Datetime"] = df_long["Date"] + pd.to_timedelta(df_long["Hour_num"], unit="h")
df_long.sort_values(by=["SegmentID", "Datetime"], inplace=True)
# print("Unique dates:", df_long["Date"].nunique())
# print("Unique segments:", df_long["SegmentID"].nunique())
segment_lengths = df_long.groupby("SegmentID")["Datetime"].count()
print(segment_lengths.describe())

n_steps = 24  # sequence length (1 day of hourly data)
X, y = [], []

for seg_id, group in df_long.groupby("SegmentID"):
    data = group["Traffic_Count"].values
    for i in range(len(data) - n_steps):
        X.append(data[i:i+n_steps])
        y.append(data[i+n_steps])  # next-hour prediction

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)
df_long.head()


Missing values:
 ID                 0
SegmentID          0
Roadway Name       0
From               0
To                 0
Direction          0
Date               0
12:00-1:00 AM      4
1:00-2:00AM        4
2:00-3:00AM        4
3:00-4:00AM        4
4:00-5:00AM        4
5:00-6:00AM        4
6:00-7:00AM        4
7:00-8:00AM        4
8:00-9:00AM        4
9:00-10:00AM       4
10:00-11:00AM      3
11:00-12:00PM      1
12:00-1:00PM     253
1:00-2:00PM      253
2:00-3:00PM      253
3:00-4:00PM      253
4:00-5:00PM      253
5:00-6:00PM      253
6:00-7:00PM      253
7:00-8:00PM      253
8:00-9:00PM      253
9:00-10:00PM     253
10:00-11:00PM    253
11:00-12:00AM    253
dtype: int64

Data types:
 ID                int64
SegmentID         int64
Roadway Name     object
From             object
To               object
Direction        object
Date             object
12:00-1:00 AM    object
1:00-2:00AM      object
2:00-3:00AM      object
3:00-4:00AM      object
4:00-5:00AM      object
5:00-6:00AM     

,ID,SegmentID,Roadway Name,From,To,Direction,Date,Hour,Traffic_Count,Hour_num,Datetime


In [ ]:
# start training an RNN